<a href="https://colab.research.google.com/github/shin-noda/leetcode-neetcode-250/blob/main/Problem355.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from collections import deque

class Twitter:
    def __init__(self):
        self.counter = 0

        # This stores (userId, [tweetIds]) for all tweets
        self.tweets = {}

        # Key (userId) -> list of UserIds that the user is following
        self.users = {}

        # Key (userId) -> top 10 tweets including the user's tweets
        self.feeds = {}


    def _checkUser(self, userId):
        if userId not in self.users:
            self.tweets[userId] = []

            # Note: [[followers (being followed)], [foloowees (following)]]
            self.users[userId] = [set(), set()]
            self.feeds[userId] = deque()


    def _updateFeeds(self, userId, tweetId, counter):
        # Update the feeds for the user himself
        if len(self.feeds[userId]) >= 10:
            # Pop the oldest tweet
            self.feeds[userId].pop()

        self.feeds[userId].appendleft((tweetId, counter))

        print(self.feeds)
        print(self.feeds[userId])
        print(self.users[userId])

        # Update the feeds for the followers
        followers = self.users[userId][0]
        for follower in followers:
            if len(self.feeds[follower]) >= 10:
                self.feeds[follower].pop()

            self.feeds[follower].appendleft((tweetId, counter))


    def _mergeFeeds(self, userId, newUserId):
        curr = set(list(self.feeds[userId]))
        new_tweets = list(self.tweets[newUserId])

        for i in range(len(new_tweets)):
            curr.add(new_tweets[i])

        curr_list = list(curr)

        # Sort by the counter
        curr_list.sort(key=lambda x: x[1], reverse=True)
        self.feeds[userId] = deque(curr_list)


    def _deleteFeeds(self, userId, unfollowId):
        curr = list(self.feeds[userId])

        deleting_tweets = set()
        dts = list(self.tweets[unfollowId])

        for i in range(len(dts)):
            deleting_tweets.add(dts[i][0])

        tmp = []
        for i in range(len(curr)):
            feed, counter = curr[i]
            if feed not in deleting_tweets:
                tmp.append((feed, counter))

        tmp.sort(key=lambda x: x[1], reverse=True)
        self.feeds[userId] = deque(tmp)


    def postTweet(self, userId, tweetId):
        print("Post")

        # Increment the counter
        self.counter += 1

        self._checkUser(userId)

        # Update tweets for the user himself
        self.tweets[userId].append((tweetId, self.counter))

        # Update feeds
        self._updateFeeds(userId, tweetId, self.counter)

        print()


    def getNewsFeed(self, userId):
        print("Get")

        print(self.feeds)

        if userId not in self.feeds:
            return []

        feeds = list(self.feeds[userId])

        print(feeds)
        print()

        if len(feeds) > 10:
            # Fetch the first 10 feeds
            feeds = feeds[:10]

        results = []
        for i in range(len(feeds)):
            results.append(feeds[i][0])

        return results

    def follow(self, followerId, followeeId):
        # Follower (user)
        # Followee (user is gonna follow)
        print("Follow")

        self._checkUser(followerId)
        self._checkUser(followeeId)

        # Update the (user's) account
        self.users[followerId][1].add(followeeId)

        # Update the followee's account
        self.users[followeeId][0].add(followerId)

        # Merge feeds for the follower
        self._mergeFeeds(followerId, followeeId)

        print()


    def unfollow(self, followerId, followeeId):
        print("Unfollow")

        self._checkUser(followerId)
        self._checkUser(followeeId)

        if followeeId in self.users[followerId][1]:
            # Update the follower (user's) account
            self.users[followerId][1].remove(followeeId)

        if followerId in self.users[followeeId][0]:
            # Update the followee's account
            self.users[followeeId][0].remove(followerId)

        # Delete feeds for the user
        self._deleteFeeds(followerId, followeeId)

        print()

In [ ]:
# Your Twitter object will be instantiated and called as such:
# obj = Twitter()
# obj.postTweet(userId,tweetId)
# param_2 = obj.getNewsFeed(userId)
# obj.follow(followerId,followeeId)
# obj.unfollow(followerId,followeeId)